# 1- Importing Libraries

In [1]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk
! pip install xgboost
! pip install seaborn
! pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# 2- Loading Dataset

In [4]:
df = pd.read_csv("Reviews.csv")
df = df.sample(25000)

# 3- Understanding Dataset

In [4]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [31]:
df.shape

(200000, 10)

In [5]:
df.nunique()
df["Score"].unique()

array([4, 5, 1, 2, 3])

In [6]:
df['HelpfulnessNumerator']

166712     2
126874     3
149577     0
19114      1
441329     0
          ..
524102     0
179952     0
433256     0
389765    18
450466     0
Name: HelpfulnessNumerator, Length: 50000, dtype: int64

# 4- Preprocessing the Dataset

In [5]:
df.isnull().sum()

Id                        0
ProductId                 0
UserId                    0
ProfileName               2
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   3
Text                      0
dtype: int64

In [5]:
df.dropna(inplace=True)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [8]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [9]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 24999/24999 [00:10<00:00, 2364.11it/s]


# 6- Training Machine Learning Models Using Count Vectorizer as Technique

In [16]:
X = text_preprocessed
y = df["Sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42
)

pipe = Pipeline([
    ('CountVec', CountVectorizer()),
    ('xgb', XGBClassifier(
        n_estimators=100,     # faster
        max_depth=5,
        learning_rate=0.15,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        n_jobs=-1
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [17]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

82.33333333333334
78.07590480034557
              precision    recall  f1-score   support

           0       0.78      0.33      0.46      1100
           1       0.48      0.08      0.13       551
           2       0.83      0.99      0.90      5849

    accuracy                           0.82      7500
   macro avg       0.70      0.46      0.50      7500
weighted avg       0.80      0.82      0.78      7500

[[ 360   23  717]
 [  47   42  462]
 [  53   23 5773]]


# 7- Training Machine Learning Models Using Tf-Idf as Technique

In [22]:
X = text_preprocessed
y = df["Sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        max_features=20000
    )),
    
    ('xgb', XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.15,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_jobs=-1,
    scale_pos_weight=1   # we'll fix differently below
)
    )
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

/Users/prabhsandhu/Downloads/Machine_Learning_Project/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:44:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [31]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

82.6
78.46196419299896
              precision    recall  f1-score   support

           0       0.79      0.34      0.47      1100
           1       0.53      0.09      0.15       551
           2       0.83      0.99      0.90      5849

    accuracy                           0.83      7500
   macro avg       0.72      0.47      0.51      7500
weighted avg       0.80      0.83      0.78      7500

[[ 371   19  710]
 [  51   47  453]
 [  49   23 5777]]


In [18]:
df

,ProductId,Text,Sentiment
95905,B005FUMNQO,I absolutely love these rice snacks. My husba...,Positive
294800,B005V9UG18,I tried the Orgain Creamy Chocolate Fudge and ...,Positive
7034,B004K30HO2,I like the DisposaKups. If I'm fixing more th...,Neutral
34577,B0018CJJ9W,Here is some better info. It is a voluntary re...,Negative
286924,B004EDZ83S,I give Tully's an AAA+ for marketing a great c...,Positive
...,...,...,...
479398,B000AQJRWG,Try not to sniff it until after you've boiled ...,Positive
548788,B001M2BM4I,This is a great morning starter!! If you love...,Positive
167985,B001AHL6CI,I really love olive tampanade and these chips ...,Positive
364012,B000G6RYOI,"Great flavor, much lower fat than regular chip...",Positive
